# Activity Recommender

A **very simple content-based recommender** for activities. You type a *desired*
activity - possibly incomplete (only a few fields) - and it returns the most similar
activities from a dataset.

**Technique** (same as `autoencoder.ipynb`): represent every activity as a numeric
vector and rank candidates by **cosine similarity** to the query. The only twist:
a partial query is scored **only on the fields the user actually filled in** (missing
fields are masked out), so an incomplete input still works.

Steps:
1. **Create dataset** - 100 synthetic rows modeled on `entities.py::ActivityEntity` (10 key features).
2. **Prepare the data** - scale numeric features, one-hot encode categorical ones.
3. **Build the recommender** - cosine similarity over the provided fields.
4. **Use the recommender** - feed an incomplete typed activity, get suggestions.

## Setup

`.venv-notebooks` is a `uv`-managed venv (no `pip` inside it), so we install with
`uv pip install` pointed at this kernel's interpreter.

In [1]:
import sys

!uv pip install --python "{sys.executable}" numpy pandas scikit-learn

Using Python 3.12.10 environment at: /home/dvorka/p/mytral/git/mytral/.venv
Audited 3 packages in 6ms


In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

print("imports ready")

imports ready


## Step 1: create the dataset

100 synthetic activities. We keep the **10 most important** features of
`ActivityEntity` (not all of them):

| Feature | Kind | From `ActivityEntity` |
|---------|------|-----------------------|
| `activity_type_key` | categorical | sport: run / ride / rowing / ski / swim |
| `intensity` | categorical | easy / long / lsd / tempo / hard / race |
| `distance_m` | numeric | `distance` (meters) |
| `duration_min` | numeric | derived from `hours/minutes/seconds` |
| `elevation_gain_m` | numeric | `elevation_gain` |
| `avg_hr` | numeric | `avg_hr` |
| `max_hr` | numeric | `max_hr` |
| `avg_speed_kmh` | numeric | `avg_speed` |
| `kcal` | numeric | `kcal` |
| `avg_cadence` | numeric | `avg_cadence` |

Values are **sport-conditioned** (a ride has bike-like speed/cadence/distance, a swim
swim-like ones) so that "similar" is meaningful. Seeded for reproducibility.

In [3]:
rng = np.random.default_rng(42)

SPORTS = ["run", "ride", "rowing", "ski", "swim"]
INTENSITIES = ["easy", "long", "lsd", "tempo", "hard", "race"]

# plausible per-sport ranges: distance (km), speed (km/h), cadence, elevation (m), HR
PROFILE = {
    "run":    {"dist": (3, 21),  "speed": (8, 15),  "cad": (150, 185), "elev": (0, 600),  "hr": (130, 178)},
    "ride":   {"dist": (20, 110),"speed": (20, 36), "cad": (80, 98),   "elev": (0, 1800), "hr": (115, 168)},
    "rowing": {"dist": (2, 18),  "speed": (10, 15), "cad": (20, 34),   "elev": (0, 0),    "hr": (140, 185)},
    "ski":    {"dist": (8, 42),  "speed": (8, 22),  "cad": (0, 0),     "elev": (100, 2200),"hr": (125, 172)},
    "swim":   {"dist": (1, 4),   "speed": (3, 6),   "cad": (28, 42),   "elev": (0, 0),    "hr": (115, 160)},
}


def _u(lo, hi):
    return lo if lo == hi else float(rng.uniform(lo, hi))


rows = []
for i in range(100):
    sport = SPORTS[i % len(SPORTS)]  # even spread across sports
    p = PROFILE[sport]
    dist_km = _u(*p["dist"])
    speed = _u(*p["speed"])
    avg_hr = _u(*p["hr"])
    duration_min = dist_km / speed * 60.0
    rows.append({
        "activity_type_key": sport,
        "intensity": INTENSITIES[int(rng.integers(len(INTENSITIES)))],
        "distance_m": round(dist_km * 1000),
        "duration_min": round(duration_min, 1),
        "elevation_gain_m": round(_u(*p["elev"])),
        "avg_hr": round(avg_hr, 1),
        "max_hr": round(min(200.0, avg_hr + _u(8, 25)), 1),
        "avg_speed_kmh": round(speed, 1),
        "kcal": round(duration_min * _u(8, 13)),
        "avg_cadence": round(_u(*p["cad"]), 1),
    })

df = pd.DataFrame(rows)
print("dataset shape:", df.shape)
df.head()

dataset shape: (100, 10)


,activity_type_key,intensity,distance_m,duration_min,elevation_gain_m,avg_hr,max_hr,avg_speed_kmh,kcal,avg_cadence
0,run,easy,16931,91.8,57,171.2,195.8,11.1,1083,177.5
1,ride,hard,31530,69.5,1668,134.7,153.6,27.2,842,88.0
2,rowing,race,5636,26.5,0,142.9,161.6,12.8,312,25.0
3,ski,hard,41004,120.0,509,161.6,177.5,20.5,986,0.0
4,swim,lsd,1463,17.4,0,148.5,162.1,5.0,171,34.6


## Step 2: prepare the data

Turn each activity into a numeric vector so cosine similarity is meaningful:

- **numeric** features -> min-max scaled to `[0, 1]` (so no single big-unit feature,
  e.g. kcal, dominates the distance),
- **categorical** features -> one-hot columns.

We keep a `col_index` map (feature -> which matrix columns it owns) so a **partial
query** can activate only the columns for the fields the user provided.

In [4]:
NUMERIC = [
    "distance_m", "duration_min", "elevation_gain_m", "avg_hr",
    "max_hr", "avg_speed_kmh", "kcal", "avg_cadence",
]
CATEGORICAL = {
    "activity_type_key": SPORTS,
    "intensity": INTENSITIES,
}

# min/max for scaling, learned from the dataset
mins, maxs = df[NUMERIC].min(), df[NUMERIC].max()

# column layout: one column per numeric feature, then a one-hot block per categorical
col_index, cursor = {}, 0
for f in NUMERIC:
    col_index[f] = [cursor]
    cursor += 1
for f, vocab in CATEGORICAL.items():
    col_index[f] = list(range(cursor, cursor + len(vocab)))
    cursor += len(vocab)
D = cursor  # total vector width


def _scale(f, x):
    lo, hi = mins[f], maxs[f]
    return 0.0 if hi == lo else float(np.clip((x - lo) / (hi - lo), 0.0, 1.0))


def vectorize(record):
    """Encode a full or partial activity as (vector, active_mask).

    active_mask marks the columns the record actually provides, so a partial
    query is later compared only on the fields the user typed.
    """
    vec = np.zeros(D)
    active = np.zeros(D, dtype=bool)
    for f in NUMERIC:
        if record.get(f) is not None:
            c = col_index[f][0]
            vec[c], active[c] = _scale(f, record[f]), True
    for f, vocab in CATEGORICAL.items():
        val = record.get(f)
        if val is not None:
            active[col_index[f]] = True  # whole one-hot block is in play
            if val in vocab:
                vec[col_index[f][vocab.index(val)]] = 1.0
    return vec, active


# full feature matrix for the 100 activities
matrix = np.vstack([vectorize(r)[0] for r in df.to_dict("records")])
print("feature matrix shape:", matrix.shape, "  (rows x vector width)")

feature matrix shape: (100, 19)   (rows x vector width)


## Step 3: build the recommender

Encode the query the same way, restrict both the query and the dataset matrix to the
**active columns** (the fields the user provided), and rank by cosine similarity.

In [5]:
def recommend(query, k=5):
    """Return the k activities most similar to a (possibly partial) query.

    Parameters
    ----------
    query : dict
        Any subset of the 10 features, e.g. {"activity_type_key": "run",
        "intensity": "tempo", "distance_m": 10000}.
    k : int
        Number of suggestions to return.
    """
    q, active = vectorize(query)
    cols = np.where(active)[0]
    if cols.size == 0:
        raise ValueError("query has no recognized features")

    sims = cosine_similarity(q[cols].reshape(1, -1), matrix[:, cols])[0]
    top = np.argsort(-sims)[:k]

    result = df.iloc[top].copy()
    result.insert(0, "similarity", sims[top].round(3))
    return result

## Step 4: use the recommender

Two **incomplete** typed queries - only a handful of fields each.

In [6]:
# "I want a ~10 km tempo run" - 3 fields only
query_1 = {
    "activity_type_key": "run",
    "intensity": "tempo",
    "distance_m": 10000,
}
print("query:", query_1)
recommend(query_1, k=5)

query: {'activity_type_key': 'run', 'intensity': 'tempo', 'distance_m': 10000}


,similarity,activity_type_key,intensity,distance_m,duration_min,elevation_gain_m,avg_hr,max_hr,avg_speed_kmh,kcal,avg_cadence
45,1.000,run,tempo,10504,65.5,220,147.6,161.2,9.6,648,174.0
80,1.000,run,tempo,6257,26.2,175,132.1,148.5,14.3,286,167.3
95,0.503,run,hard,18360,109.2,238,158.3,171.0,10.1,1357,156.6
25,0.503,run,race,19849,130.1,261,132.2,157.0,9.2,1621,176.2
50,0.503,run,hard,17312,71.0,57,142.2,160.6,14.6,629,169.8


In [7]:
# "A long, hilly ride of ~80 km" - sport + distance + elevation, no intensity
query_2 = {
    "activity_type_key": "ride",
    "distance_m": 80000,
    "elevation_gain_m": 1200,
}
print("query:", query_2)
recommend(query_2, k=5)

query: {'activity_type_key': 'ride', 'distance_m': 80000, 'elevation_gain_m': 1200}


,similarity,activity_type_key,intensity,distance_m,duration_min,elevation_gain_m,avg_hr,max_hr,avg_speed_kmh,kcal,avg_cadence
11,1.000,ride,race,79572,165.1,1196,156.5,171.5,28.9,1993,83.0
71,0.998,ride,lsd,78315,154.4,1417,131.8,149.1,30.4,1568,91.3
51,0.997,ride,tempo,71519,156.3,1375,142.7,164.3,27.5,1635,90.8
26,0.995,ride,lsd,100171,175.3,1390,142.5,161.7,34.3,1729,81.7
31,0.992,ride,hard,86509,234.9,1670,121.6,136.3,22.1,2233,88.8
